# NC Baseline Fix: Weight Decay Sweep

**Fully self-contained** — no other notebook needs to be running.

Sweeps wd in {5e-4, 1e-3, 2e-3} to find the value that produces
full NC1 collapse, and records the feature norm at collapse.

**Before running:**
- Accelerator: GPU P100
- Internet: ON

**Est. runtime: ~45 min** (3 configs x 300 epochs)

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'
print(f'GPU: {torch.cuda.get_device_name(0)}  |  PyTorch: {torch.__version__}')


GPU: Tesla P100-PCIE-16GB  |  PyTorch: 2.9.0+cu126


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:
transform_tr = T.Compose([
    T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
transform_te = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])

trainset = torchvision.datasets.CIFAR10('/kaggle/working/data',
    train=True,  download=True, transform=transform_tr)
testset  = torchvision.datasets.CIFAR10('/kaggle/working/data',
    train=False, download=True, transform=transform_te)

train_loader = DataLoader(trainset, batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Train: {len(trainset):,}  Test: {len(testset):,}')


100%|██████████| 170M/170M [00:10<00:00, 16.2MB/s]


Train: 50,000  Test: 10,000


In [3]:
class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1, act_cls=nn.ReLU):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.act   = act_cls()
        self.skip  = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c))
    def forward(self, x):
        return self.act(
            self.bn2(self.conv2(self.act(self.bn1(self.conv1(x)))))
            + self.skip(x))

class ResNetCIFAR(nn.Module):
    # depth must be 6n+2: 20->n=3, 32->n=5, 44->n=7, 56->n=9
    def __init__(self, depth=20, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0
        n = (depth - 2) // 6
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.act1   = act_cls()
        self.layer1 = self._make(16, 16, n, 1, act_cls)
        self.layer2 = self._make(16, 32, n, 2, act_cls)
        self.layer3 = self._make(32, 64, n, 2, act_cls)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)
        self._feats = None
        self.pool.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.flatten(1).detach()))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make(self, in_c, out_c, n, stride, act_cls):
        layers = [BasicBlock(in_c, out_c, stride, act_cls)]
        for _ in range(n-1):
            layers.append(BasicBlock(out_c, out_c, 1, act_cls))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        return self.fc(self.pool(x).flatten(1))

    def get_features(self, x):
        self(x); return self._feats

    def get_classifier_weights(self):
        return self.fc.weight.detach()

print('ResNetCIFAR defined.')


ResNetCIFAR defined.


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    feats_list, labels_list = [], []
    for x, y in loader:
        feats_list.append(model.get_features(x.to(DEVICE)).cpu())
        labels_list.append(y)
    H = torch.cat(feats_list).float()
    Y = torch.cat(labels_list)
    N, d = H.shape
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw = sum((H[Y==c]-mu_c[c]).T @ (H[Y==c]-mu_c[c]) for c in range(K)) / N
    Sb = M.T @ M / K
    nc1 = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    Mn  = F.normalize(M, dim=1)
    cos = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2 = (cos[mask] - (-1.0/(K-1))).abs().mean().item()
    Wn  = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3 = (1 - (Mn * Wn).sum(1).mean()).item()
    feat_norm = H.norm(dim=1).mean().item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3, 'feat_norm': feat_norm}

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

def train_nc(model, name='model', lr=0.1, wd=1e-4, epochs=300, nc_every=5):
    model = model.to(DEVICE)
    opt   = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                            weight_decay=wd, nesterov=True)
    sched = torch.optim.lr_scheduler.MultiStepLR(
                opt, milestones=[150, 225], gamma=0.1)
    rows, terminal, t0 = [], False, time.time()
    for ep in range(1, epochs+1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(x), y).backward()
            opt.step()
        sched.step()
        if ep % nc_every == 0 or ep == epochs:
            tr = evaluate(model, train_loader)
            te = evaluate(model, test_loader)
            if tr >= 0.99 and not terminal:
                terminal = True
                print(f'  [{name}] Terminal phase at epoch {ep}')
            nc = compute_nc(model, train_loader) if terminal else \
                 {'nc1': None, 'nc2': None, 'nc3': None, 'feat_norm': None}
            rows.append({'epoch': ep, 'train': tr, 'test': te, **nc})
            nc1s = f"{nc['nc1']:.4f}" if nc['nc1'] is not None else 'N/A'
            fns  = f"{nc['feat_norm']:.4f}" if nc['feat_norm'] else 'N/A'
            print(f'  ep={ep:>3} tr={tr:.3f} te={te:.3f} '
                  f'nc1={nc1s} fn={fns} t={(time.time()-t0)/60:.1f}m')
    return pd.DataFrame(rows)

print('train_nc, compute_nc, evaluate ready.')


train_nc, compute_nc, evaluate ready.


In [5]:
# Sweep wd in {5e-4, 1e-3, 2e-3} — 3 configs x 300 epochs ~ 45 min
WD_SWEEP = [5e-4, 1e-3, 2e-3]
results  = {}

for wd in WD_SWEEP:
    name = f'RN20-ReLU-wd{wd}'
    print(f'\n=== wd={wd} ===')
    torch.manual_seed(0)
    model = ResNetCIFAR(depth=20, act_cls=nn.ReLU)
    df    = train_nc(model, name=name, lr=0.1, wd=wd, epochs=300, nc_every=5)
    fname = f'wd{str(wd).replace(".","p")}.csv'
    df.to_csv(SAVE_DIR + fname, index=False)
    results[wd] = df
    nc_done = df.dropna(subset=['nc1'])
    if len(nc_done):
        print(f'  NC1 final:   {nc_done.nc1.iloc[-1]:.6f}')
        t_nc_rows = nc_done[nc_done.nc1 < 0.1]
        if len(t_nc_rows):
            t_nc = t_nc_rows.epoch.iloc[0]
            fn   = t_nc_rows.feat_norm.iloc[0]
            print(f'  T_NC:        epoch {t_nc}   feat_norm={fn:.4f}')
        else:
            print(f'  NC1 never reached 0.1 in 300 epochs')
    print(f'  Test acc:    {df.test.iloc[-1]:.4f}')

print('\nAll runs complete. CSVs saved to /kaggle/working/')



=== wd=0.0005 ===
  ep=  5 tr=0.749 te=0.735 nc1=N/A fn=N/A t=1.2m
  ep= 10 tr=0.788 te=0.769 nc1=N/A fn=N/A t=2.3m
  ep= 15 tr=0.767 te=0.748 nc1=N/A fn=N/A t=3.4m
  ep= 20 tr=0.795 te=0.777 nc1=N/A fn=N/A t=4.5m
  ep= 25 tr=0.805 te=0.789 nc1=N/A fn=N/A t=5.7m
  ep= 30 tr=0.782 te=0.762 nc1=N/A fn=N/A t=6.8m
  ep= 35 tr=0.821 te=0.801 nc1=N/A fn=N/A t=8.0m
  ep= 40 tr=0.827 te=0.816 nc1=N/A fn=N/A t=9.2m
  ep= 45 tr=0.794 te=0.786 nc1=N/A fn=N/A t=10.3m
  ep= 50 tr=0.760 te=0.735 nc1=N/A fn=N/A t=11.4m
  ep= 55 tr=0.848 te=0.830 nc1=N/A fn=N/A t=12.6m
  ep= 60 tr=0.852 te=0.836 nc1=N/A fn=N/A t=13.8m
  ep= 65 tr=0.830 te=0.814 nc1=N/A fn=N/A t=15.0m
  ep= 70 tr=0.783 te=0.757 nc1=N/A fn=N/A t=16.1m
  ep= 75 tr=0.825 te=0.811 nc1=N/A fn=N/A t=17.3m
  ep= 80 tr=0.815 te=0.799 nc1=N/A fn=N/A t=18.4m
  ep= 85 tr=0.835 te=0.812 nc1=N/A fn=N/A t=19.5m
  ep= 90 tr=0.801 te=0.786 nc1=N/A fn=N/A t=20.6m
  ep= 95 tr=0.834 te=0.816 nc1=N/A fn=N/A t=21.8m
  ep=100 tr=0.844 te=0.819 nc1=N/A fn=N

In [6]:
# Also load the original wd=1e-4 run if saved
import os
df_1e4 = None
for path in ['/kaggle/working/baseline.csv', '/kaggle/input/baseline/baseline.csv']:
    if os.path.exists(path):
        df_1e4 = pd.read_csv(path)
        print(f'Loaded baseline from {path}')
        break

plt.rcParams.update({'font.family':'serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False})

all_dfs    = ([df_1e4] if df_1e4 is not None else []) + list(results.values())
all_labels = ([f'wd=1e-4 (partial NC)'] if df_1e4 is not None else []) + \
             [f'wd={w}' for w in WD_SWEEP]
all_colors = (['#9E9E9E'] if df_1e4 is not None else []) + \
             ['#FF9800','#4CAF50','#2196F3']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) NC1 over time
ax = axes[0]
for df_, lbl, col in zip(all_dfs, all_labels, all_colors):
    nc = df_.dropna(subset=['nc1'])
    if len(nc): ax.semilogy(nc.epoch, nc.nc1, color=col, lw=2, label=lbl)
ax.axhline(0.1, color='black', ls=':', lw=1, label='NC1=0.1 threshold')
ax.set(xlabel='Epoch', ylabel='NC1 (log)', title='(a) NC1 collapse by weight decay')
ax.legend(fontsize=8); ax.grid(alpha=0.25)

# (b) Feature norm over time
ax = axes[1]
for df_, lbl, col in zip(all_dfs, all_labels, all_colors):
    nc = df_.dropna(subset=['nc1'])
    if len(nc): ax.plot(nc.epoch, nc.feat_norm, color=col, lw=2, label=lbl)
ax.set(xlabel='Epoch', ylabel='Mean feature norm', title='(b) Feature norm by weight decay')
ax.legend(fontsize=8); ax.grid(alpha=0.25)

# (c) T_NC and feature norm at collapse vs wd
ax = axes[2]
wd_done, t_nc_done, fn_done = [], [], []
for wd, df_ in results.items():
    nc = df_.dropna(subset=['nc1'])
    rows = nc[nc.nc1 < 0.1]
    if len(rows):
        wd_done.append(wd)
        t_nc_done.append(rows.epoch.iloc[0])
        fn_done.append(rows.feat_norm.iloc[0])
if wd_done:
    ax.semilogx(wd_done, t_nc_done, 'o-', color='#4CAF50', lw=2, ms=8, label='T_NC (epoch)')
    ax2 = ax.twinx()
    ax2.semilogx(wd_done, fn_done, 's--', color='#E91E63', lw=2, ms=8,
                label='feat_norm at T_NC')
    ax2.set_ylabel('Feature norm at T_NC', color='#E91E63')
    ax.set(xlabel='Weight decay lambda', ylabel='T_NC (epoch)',
           title='(c) Collapse speed vs lambda')
    ax.legend(loc='upper left', fontsize=9)
    ax2.legend(loc='upper right', fontsize=9)
    print('Feature norm at T_NC across lambda values:')
    for wd, fn in zip(wd_done, fn_done):
        print(f'  wd={wd}: feat_norm={fn:.4f}')
    cv = np.std(fn_done) / np.mean(fn_done)
    print(f'  mean={np.mean(fn_done):.4f}  std={np.std(fn_done):.4f}  CV={cv:.3f}')
    if cv < 0.15:
        print('  => Feature norm threshold is CONSISTENT across lambda (CV<15%)')
        print('  => This is your paper central result!')
    else:
        print('  => Feature norm varies with lambda (CV>=15%)')
else:
    ax.text(0.5, 0.5, 'No T_NC reached yet', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    ax.set(title='(c) Collapse speed vs lambda')

fig.suptitle('Weight Decay Sweep: NC Dynamics | ResNet-20 | ReLU | CIFAR-10',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_wd_sweep_nc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {SAVE_DIR}fig_wd_sweep_nc.png')


Saved: /kaggle/working/fig_wd_sweep_nc.png
